# Quantizing the Facebook denoisers (dns48, dns64, master64)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadRaziei/libspeech/blob/main/labs/quantize_denoiser/quantize_denoiser.ipynb)

Makes int8 versions of a Facebook denoiser and measures what they cost, in three parts:

1. **Post-training quantization** (CPU, minutes): calibrate on real noisy recordings, quantize, compare with fp32.
2. **Teacher-student fine-tuning** (GPU recommended): train a weight-quantized student to reproduce the fp32 teacher, then quantize it.
3. **Through libspeech**: run the fp32 and int8 models in libspeech itself, time them, and listen.

Quality is measured as dB of agreement with the fp32 model's own output (there is no clean reference for real noisy
recordings), and speed as the real-time factor (processing time / audio time, lower is faster) on the machine running the notebook.
Set the runtime to GPU for Part 2 (Runtime > Change runtime type).

## 0. Settings

In [ ]:
MODEL = "dns48"      # dns48 | dns64 | master64
DATA_DIR = None      # a folder of recordings that are noisy already; None downloads two samples from libspeech's release
STEPS = 3000         # distillation steps in Part 2
SMOKE = False        # True: tiny random model, synthetic data, a few steps - only checks that the notebook runs end to end
REPO = "https://github.com/MohammadRaziei/libspeech.git"
BRANCH = "main"

## 1. Install
libspeech is built from source, which takes a few minutes the first time.

In [ ]:
!pip install -q "git+https://github.com/MohammadRaziei/libspeech.git"
!pip install -q onnx onnxruntime soundfile scipy
!pip install -q --no-deps denoiser

## 2. Get the lab's scripts and some audio

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("onnx_ptq.py").exists():  # not running from the lab folder: clone the repo and go there
    if not Path("libspeech").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO], check=True)
    os.chdir("libspeech/labs/quantize_denoiser")
sys.path.insert(0, os.getcwd())

import lab_helpers as lab
import onnx_ptq
from mixtures import MixtureSource, NoisyRecordings

if SMOKE:
    DATA_DIR = None
elif DATA_DIR is None:
    DATA_DIR = lab.sample_recordings("recordings")
TEACHER = lab.teacher_onnx("random:8" if SMOKE else MODEL, "work")  # the fp32 model every result is compared with
print("teacher:", TEACHER, f"({Path(TEACHER).stat().st_size / 1e6:.1f} MB)")


def source(seed, seconds=4.0):
    if DATA_DIR:
        return NoisyRecordings(DATA_DIR, seconds=seconds, seed=seed)
    return MixtureSource(seconds=seconds, seed=seed)


# Calibration and evaluation clips use different seeds, but come from the same files.
calibration, _ = source(0).batch(3 if SMOKE else 8)
held_out, _ = source(1234).batch(2 if SMOKE else 8)

## Part 1 - post-training int8

Static int8 for the convolutions (per-channel weights, activations calibrated on the clips above) and dynamic int8 for the LSTM.
`agreement` is how many dB the quantized output stays above its own error against the fp32 output (higher is closer).

In [ ]:
onnx_ptq.quantize(TEACHER, "work/ptq_int8.onnx", calibration, method="percentile")
ptq = onnx_ptq.evaluate(TEACHER, "work/ptq_int8.onnx", held_out)
ptq

### Which layers hurt? (optional, slow)

Quantizes one convolution at a time, and the LSTM on its own, and reports the agreement with fp32 for each. The slowest
layers to quantize are the ones to keep in fp32. This repeats the calibration for every layer, so it takes a while.

In [ ]:
RUN_SENSITIVITY = False
KEEP_FP32 = 3

if RUN_SENSITIVITY:
    rows = onnx_ptq.sensitivity(
        TEACHER, calibration, held_out, method="percentile", per_channel=True, workdir="work/sensitivity"
    )
    keep = [r["node"] for r in rows if r["node"] != "LSTM"][:KEEP_FP32]
    onnx_ptq.quantize(TEACHER, "work/ptq_mixed.onnx", calibration, exclude=keep, method="percentile")
    mixed = onnx_ptq.evaluate(TEACHER, "work/ptq_mixed.onnx", held_out)
    print("kept in fp32:", keep)
    mixed

## Part 2 - teacher-student fine-tuning

The student is a copy of the fp32 model whose weights are rounded to the int8 grid on every forward pass; it is trained to
reproduce the teacher's output on the recordings. The teacher's output is the label, so no clean speech and none of
the original training data are needed. Afterwards the same quantizer as in Part 1 turns the student into an int8 model: its
weights are already on the grid, so only the activations are calibrated.

Needs the original PyTorch weights (downloaded by the `denoiser` package) and is slow without a GPU.

In [ ]:
import distill_qat
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("training on", device)
args = [
    "--model", "random:8" if SMOKE else MODEL,
    "--out", "work/qat",
    "--device", device,
    "--steps", "20" if SMOKE else str(STEPS),
    "--seconds", "1" if SMOKE else "3",
    "--batch", "2" if SMOKE else "8",
    "--eval-clips", "2" if SMOKE else "16",
]
if DATA_DIR:
    args += ["--noisy-dir", DATA_DIR]
distill_qat.main(args)

In [ ]:
onnx_ptq.quantize("work/qat/student_grid_fp32.onnx", "work/qat_int8.onnx", calibration, method="percentile")
qat = onnx_ptq.evaluate(TEACHER, "work/qat_int8.onnx", held_out)
print(lab.table({"post-training (Part 1)": ptq, "fine-tuned (Part 2)": qat}))

## Part 3 - through libspeech

libspeech runs the models with its own bundled ONNX Runtime, which can be a different (older) version from the `onnxruntime`
package used above, so speed can differ between the two. Both are reported.

In [ ]:
import libspeech

clip = held_out[0]
names = {
    "fp32": lab.register_in_libspeech(TEACHER, "lab-fp32.onnx"),
    "post-training int8": lab.register_in_libspeech("work/ptq_int8.onnx", "lab-ptq-int8.onnx"),
    "fine-tuned int8": lab.register_in_libspeech("work/qat_int8.onnx", "lab-qat-int8.onnx"),
}
outputs, results = {}, {}
for label, name in names.items():
    outputs[label], rtf = lab.run_in_libspeech("lab-fp32.onnx" if label == "fp32" else name, clip)
    results[label] = {"rtf (libspeech)": rtf, "agreement with fp32 (dB)": lab.agreement(outputs[label], outputs["fp32"]) if label != "fp32" else float("nan")}
print(lab.table(results))
print("libspeech's ONNX Runtime:", libspeech.ONNXRUNTIME_LIB_PATH)

In [ ]:
lab.listen("noisy input", clip)
for label, audio in outputs.items():
    lab.listen(label, audio)

## Save the results

In [ ]:
import shutil

shutil.make_archive("quantized_models", "zip", "work", ".")
try:
    from google.colab import files

    files.download("quantized_models.zip")
except ImportError:
    print("saved quantized_models.zip")